In [1]:
%load_ext autoreload

In [ ]:
%autoreload 2
import torch
import numpy as np
from toy_model import Tree, TreeDataset
import json
import torch.nn.functional as F
import matplotlib.pyplot as plt


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tree_dict = json.load(open('./tree.json', 'r'))
tree = Tree(tree_dict=tree_dict)

D_MODEL = tree.n_features

matryoshka_config = {
    'n_latents': tree.n_features,
    'target_l0': 1.36,  # L0 of the true features
    'n_prefixes': 5,
    'd_model': D_MODEL,
    'n_steps': 15_000, 
    'lr': 3e-2,
    'permute_latents': True,
    'sparsity_type': 'l1',
    'starting_sparsity_loss_scale': 0.1
}

mp_config = {
    'd_model': D_MODEL,
    'n_latents': 20,
    'threshold':0.05,
    'n_steps': matryoshka_config['n_steps'],
    'lr': 3e-2,
}
topk_config = {
    'd_model': D_MODEL,
    'n_latents': 20,
    'n_steps': matryoshka_config['n_steps'],
    'lr': 3e-2,
    'target_l0': 3,
}

vanilla_config = matryoshka_config | {'n_prefixes': 1, 'permute_latents': False}

In [4]:
from scipy.optimize import linear_sum_assignment

@torch.no_grad()
def get_latent_perm(sae, tree_ds, include_all_latents=False):
    '''
    Get permutation of sae latents that tries to assign each latent to its closest-matching ground-truth feature
    H.T. Julian D'Costa for the linear_sum_assignment idea here.
    '''
    global DEVICE
    sample = tree_ds.tree.sample(1000).to(DEVICE)
    x = sample @ tree_ds.true_feats.to(DEVICE)
    
    feat_norms = tree_ds.true_feats.norm(dim=-1).to(DEVICE)
    scaled_sample = sample * feat_norms[None, :]

    true_acts = scaled_sample
    true_acts = true_acts/true_acts.max(dim=0, keepdim=True).values.clamp(min=1e-10)

    with torch.no_grad():
        sae_acts = sae.get_acts(x)
        sae_acts = sae_acts/sae_acts.max(dim=0, keepdim=True).values.clamp(min=1e-10)
        sims = (sae_acts.T @ true_acts).cpu()

    max_value = sims.max()
    cost_matrix = max_value - sims

    row_ind, col_ind = row_ind, col_ind = linear_sum_assignment(cost_matrix.detach().cpu().numpy().T)
    leftover_features = np.array(list({i for i in range(sims.shape[0]) if sae_acts.max(dim=0).values[i] > 0} - set(col_ind))).astype(int)
    
    if include_all_latents:
        return torch.tensor(np.concatenate([col_ind, leftover_features]))
    else:
        return torch.tensor(col_ind)

In [5]:
# Hierarchical mixing coefficients controlling feature correlation at each depth level

# epsilon_childrens controls similarity between sibling (child) features
epsilon_childrens = [0, 0.0895, 0.163, 0.226, 0.282, 0.3335, 0.3825, 0.431, 0.482]

# epsilon_parents controls similarity between parent-level features
epsilon_parents = [0, 0.3, 0.44, 0.529, 0.594, 0.6467, 0.692, 0.733, 0.7739]

In [6]:
l = 1  # Correlation 0.1

In [7]:
from torch.utils.data import DataLoader
from tqdm import tqdm

# Generate and normalize true latent features
true_feats = (torch.randn(tree.n_features, D_MODEL) / np.sqrt(D_MODEL)).to(DEVICE)

# Create orthonormal basis
Q, _ = torch.linalg.qr(torch.randn(D_MODEL, D_MODEL))
Q_copy = Q.clone()

# Function to mix indices with given epsilon
def mix_indices(indices, epsilon):
    for i in indices:
        others = [j for j in indices if j != i]
        update = (1 - epsilon) * Q_copy[i]
        update += (epsilon / len(others)) * sum(Q_copy[j] for j in others)
        Q[i] = update

# Apply parent and child mixing
mix_indices([0, 4, 8, 12, 13, 14, 15, 16, 17, 18, 19], epsilon_parents[l])
mix_indices([1, 2, 3], epsilon_childrens[l])
mix_indices([5, 6, 7], epsilon_childrens[l])
mix_indices([9, 10, 11], epsilon_childrens[l])

# Re-normalize features
Q = Q / torch.norm(Q, dim=1, keepdim=True)
true_feats = Q[:tree.n_features].to(DEVICE)

# Create dataset and dataloader
dataset = TreeDataset(tree, true_feats.cpu(), batch_size=200, num_batches=vanilla_config["n_steps"])
dataloader = DataLoader(dataset, batch_size=None, num_workers=6, pin_memory=True)

# Reference batch for visualization
ref_acts = tree.sample(100).to(DEVICE)
ref_acts *=  1.5 + 0.25 * torch.randn(ref_acts.shape, device=DEVICE)
ref_x = ref_acts @ dataset.true_feats.to(DEVICE)

In [ ]:
d = true_feats.detach().cpu()
plt.imshow(d@d.T, cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar()
plt.axis("off");

In [ ]:
%autoreload 2
from IPython.display import clear_output
from heatmap import heatmap
from sae import MatryoshkaSAE,MatchingPursuitSAE, BatchTopKSAE

vanilla_sae = MatryoshkaSAE(**vanilla_config).to(DEVICE)
matryoshka_sae = MatryoshkaSAE(**matryoshka_config).to(DEVICE)
mp_sae = MatchingPursuitSAE(**mp_config).to(DEVICE)
topk_sae = BatchTopKSAE(**topk_config).to(DEVICE)

loss_matryoshka = []
loss_vanilla = []
loss_mp = []
loss_topk= []

for step, batch in tqdm(enumerate(dataloader), total=vanilla_config['n_steps']):
    batch = batch.to(DEVICE)

    matryoshka_sae.step(batch)
    vanilla_sae.step(batch)
    mp_sae.step(batch)
    topk_sae.step(batch)
    
    
    if (step+1)% 1000== 0:
        topk_sae.target_l0 = 1.36  #Reduce target l0 to 1.36 for batch topk
        
        clear_output(wait=True)
    
        heatmap(ref_acts.cpu(), title='Ground-Truth Features').show()

        matryoshka_perm = get_latent_perm(matryoshka_sae, dataset)
        heatmap(matryoshka_sae.get_acts(ref_x)[:,matryoshka_perm].cpu(), title=f'Matryoshka Latents  |  Sparsity Reg: {matryoshka_sae.sparsity_controller():.2f}  |  Step {step}',).show()

        vanilla_perm = get_latent_perm(vanilla_sae, dataset)
        heatmap(vanilla_sae.get_acts(ref_x)[:,vanilla_perm].cpu(), title=f'Vanilla Latents  |  Sparsity Reg: {vanilla_sae.sparsity_controller():.2f}  |  Step {step}').show()
        
        topk_perm = get_latent_perm(topk_sae, dataset)
        heatmap(topk_sae.get_acts(ref_x)[:,topk_perm].cpu(), title=f'Batch TopK Latents  |  Step {step}').show()
        
        mp_perm = get_latent_perm(mp_sae, dataset)
        heatmap(mp_sae.get_acts(ref_x)[:,mp_perm].cpu(), title=f'MP Latents  |  Step {step}').show()

In [ ]:
matryoshka_perm = get_latent_perm(matryoshka_sae, dataset)
vanilla_perm = get_latent_perm(vanilla_sae, dataset)
mp_perm = get_latent_perm(mp_sae, dataset)
topk_perm = get_latent_perm(topk_sae, dataset)

# Compute cosine similarity matrix of ground truth features
gt_cosine = F.normalize(true_feats, dim=1) @ F.normalize(true_feats, dim=1).T
gt_sims = heatmap(gt_cosine.cpu(), title='Ground Truth Feature Cosine Similarity', dim_names=('True Feature', 'True Feature'))
gt_sims.show()

#Compute cosine similarity between learned and ground truth features
matryoshka_feats = matryoshka_sae.W_dec.data
vanilla_feats = vanilla_sae.W_dec.data
topk_feats = topk_sae.W_dec.data
mp_feats = mp_sae.W.data

matryoshka_cosine = F.normalize(matryoshka_feats, dim=1)[matryoshka_perm] @ F.normalize(true_feats, dim=1).T
vanilla_cosine = F.normalize(vanilla_feats, dim=1)[vanilla_perm] @ F.normalize(true_feats, dim=1).T
mp_cosine = F.normalize(mp_feats, dim=1)[mp_perm] @ F.normalize(true_feats, dim=1).T
topk_cosine = F.normalize(topk_feats, dim=1)[topk_perm] @ F.normalize(true_feats, dim=1).T

m_dec_sims = heatmap(matryoshka_cosine.cpu(), title='Matryoshka Decoder, Ground-Truth Cosine Similarity', dim_names=('Matryoshka', 'True Feature'))
m_dec_sims.show()

v_dec_sims = heatmap(vanilla_cosine.cpu(), title='Vanilla Decoder, Ground-Truth Cosine Similarity', dim_names=('Vanilla', 'True Feature'))
v_dec_sims.show()

topk_dec_sims = heatmap(topk_cosine.cpu(), title='BatchTopK Decoder, Ground-Truth Cosine Similarity', dim_names=('BatchTopK', 'True Feature'))
topk_dec_sims.show()

mp_dec_sims = heatmap(mp_cosine.cpu(), title='MP Dictionary, Ground-Truth Cosine Similarity', dim_names=('MP', 'True Feature'))
mp_dec_sims.show()


#Compare encoder and decoder weights
matryoshka_enc = matryoshka_sae.W_enc.data.T
vanilla_enc = vanilla_sae.W_enc.data.T
topk_enc = topk_sae.W_enc.data.T

matryoshka_enc_true = F.normalize(matryoshka_enc, dim=1)[matryoshka_perm] @ F.normalize(true_feats, dim=1).T
vanilla_enc_true = F.normalize(vanilla_enc, dim=1)[vanilla_perm] @ F.normalize(true_feats, dim=1).T
topk_enc_true = F.normalize(topk_enc, dim=1)[topk_perm] @ F.normalize(true_feats, dim=1).T

m_enc_sims = heatmap(matryoshka_enc_true.cpu(), title='Matryoshka Encoder, Ground-Truth Cosine Similarity', dim_names=('Latent', 'Feature'))
m_enc_sims.show()
v_enc_sims = heatmap(vanilla_enc_true.cpu(), title='Vanilla Encoder, Ground-Truth Cosine Similarity', dim_names=('Latent', 'Feature'))
v_enc_sims.show()
t_enc_sims = heatmap(topk_enc_true.cpu(), title='BatchTopK Encoder, Ground-Truth Cosine Similarity', dim_names=('Latent', 'Feature'))
t_enc_sims.show()


In [ ]:
# Compute cosine similarity matrix of learned features

matryoshka_cosine = F.normalize(matryoshka_feats, dim=1)[matryoshka_perm] @F.normalize(matryoshka_feats, dim=1)[matryoshka_perm].T
vanilla_cosine = F.normalize(vanilla_feats, dim=1)[vanilla_perm] @ F.normalize(vanilla_feats, dim=1)[vanilla_perm].T
mp_cosine = F.normalize(mp_feats, dim=1)[mp_perm] @ F.normalize(mp_feats, dim=1)[mp_perm].T
topk_cosine = F.normalize(topk_feats, dim=1)[topk_perm] @ F.normalize(topk_feats, dim=1)[topk_perm].T

m_dec_sims = heatmap(torch.relu(matryoshka_cosine.cpu()), title='Matryoshka Decoder Correlation', dim_names=('Matryoshka', 'Matryoshka'))
m_dec_sims.show()

v_dec_sims = heatmap(torch.relu(vanilla_cosine.cpu()), title='Vanilla Decoder Correlation', dim_names=('Vanilla', 'Vanilla'))
v_dec_sims.show()

t_dec_sims = heatmap(torch.relu(vanilla_cosine.cpu()), title='Batch TopK Decoder Correlation', dim_names=('BatchTopK', 'BatchTopK'))
t_dec_sims.show()

mp_dec_sims = heatmap(mp_cosine.cpu(), title='MP Dictionary Correlation', dim_names=('MP', 'MP'))
mp_dec_sims.show()